# Exercise 1: The ONNX Graph of the GTK Transformer and the Split

## Objectives
In this exercise, you will:
1. Read an ONNX model: its inputs, outputs, operations and weights
2. Find, from the structure of the graph, where to cut the model
3. Run the model with ONNX Runtime and look at its output
4. Split the model into the encoder and the pair classifier, and check that the result does not change
5. Save what you need for Exercise 2

## Instructions
Complete the code cells marked with `# TODO` comments. Follow the hints provided.

## Part 1: Environment Setup
Run the following cells to set up the environment (no changes needed). <br>
No Xilinx tool is needed for this exercise: start Jupyter after `HI_CONDA_ENV=hybrid-tutorial source env.sh py`.

In [ ]:
# Load libraries and import packages (no changes needed)

# General imports
import sys
from itertools import combinations_with_replacement
from pathlib import Path

# Numpy and plotting
import numpy as np
import matplotlib.pyplot as plt

# ONNX, to read and edit the model, and ONNX Runtime, to run it
import onnx
import onnxruntime
from onnx import TensorProto, helper, numpy_helper

# Project folders: the project is the folder with hi_common.py, this exercise is in its school/ folder
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "hi_common.py").exists():
        PROJECT_DIR = folder
        break
SCHOOL_DIR = PROJECT_DIR / "school"
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(SCHOOL_DIR))

# Helpers of the project: configuration, events
import hi_common

# Local file with some useful methods (plots, helpers)
from utils import *

# Configuration, trained model and output folders
CONFIG = hi_common.load_config(PROJECT_DIR / "config" / "hybrid_small_data.yaml")
TRAINED_MODEL = PROJECT_DIR / "artifacts" / "ckpt-e7c02308" / "onnx" / "full_current.onnx"
WORK_DIR = SCHOOL_DIR / "work"
ONNX_DIR = WORK_DIR / "onnx"          # encoder.onnx and pair_head.onnx
DATA_DIR = WORK_DIR / "data"          # what Exercise 2 needs
FIGURE_DIR = WORK_DIR / "figures"
for folder in (ONNX_DIR, DATA_DIR, FIGURE_DIR):
    folder.mkdir(parents=True, exist_ok=True)
THRESHOLD = CONFIG["operating_threshold"]   # a pair is an edge when its probability p1 is above this - DO NOT CHANGE
model_config, _ = hi_common.model_config(CONFIG)

use_course_style()
print(f"onnx {onnx.__version__}, onnxruntime {onnxruntime.__version__}")
print("Environment setup correctly!")

In Exercise 0 you found that _hls4ml_ cannot convert the trained model as it is, but that it can convert its **pair classifier**. So we split the model: the encoder stays on the CPU, the pair classifier goes to the FPGA. In this exercise you do the split, working directly on the ONNX file.

## Part 2: Inside the ONNX Model
An ONNX file holds a **graph**:
- the **nodes** are the operations (`MatMul`, `Add`, `Softmax`, ...); each node reads some tensors and writes others, all identified by their names
- the **inputs** and **outputs** of the graph are the tensors the user gives and gets back
- the **initializers** are the tensors stored in the file: the weights and biases learned in training, and some constants

### Task 2.1: Load and Check the Model

In [ ]:
# TODO: load the trained model and check it

# Hint: onnx.load(path) reads an ONNX file and returns the model
# Hint: onnx.checker.check_model(model, full_check=True) validates it (types, shapes, operator versions):
#       it raises an exception if something is wrong

model = None  # Replace with correct code

# YOUR CODE HERE: check the model

for graph_input in model.graph.input:
    print(f"Input       : {graph_input.name} {shape_of(graph_input)}")
for graph_output in model.graph.output:
    print(f"Output      : {graph_output.name} {shape_of(graph_output)}")
print(f"Nodes       : {len(model.graph.node)}")
print(f"Initializers: {len(model.graph.initializer)}")

The model takes the hits of one event, `data [num_nodes, 4]` (x, y, GTK station, time), and the list of the pairs to score, `pair_indices [num_pairs, 2]`. It returns, for every pair, the probabilities of "no edge" and "edge", `output [1, num_pairs, 2]`. `num_nodes` and `num_pairs` are **dynamic** axes: the same graph works for events of any size.

Here are the first nodes of the graph. Look at how they are linked: the output of a node is the input of the next ones.

In [ ]:
# Print the first nodes of the graph (no changes needed)
for node in model.graph.node[:8]:
    print(f"{node.op_type:10s} inputs: {list(node.input)}")
    print(f"{'':10s} outputs: {list(node.output)}")

### Task 2.2: Who Computes What?
To move around the graph we need to know, for every tensor, which node computes it. Build this map as a Python dictionary.

In [ ]:
# TODO: build a dictionary that gives, for every tensor name, the node that computes it

# Hint: loop over model.graph.node: every node has the list of the names of the tensors it writes, node.output
# Hint: a node can write more than one tensor: add all of them

producer_of = {}  # tensor name -> the node that computes it

# YOUR CODE HERE

output_name = model.graph.output[0].name
print(f"{len(producer_of)} tensors are computed by the {len(model.graph.node)} nodes")
print(f"The model output '{output_name}' is computed by the node '{producer_of[output_name].name}' "
      f"({producer_of[output_name].op_type})")

## Part 3: Find Where to Cut
This is what we want to obtain: the model goes from working **per hit** (the encoder) to working **per pair** (the pair building and the pair classifier). We cut it at two tensors:
- **cut 1**, the **embeddings** `[1, N, 32]`: the output of the encoder, 32 numbers for each hit
- **cut 2**, the **pair features** `[1, K, 64]`: for each pair, the embeddings of its two hits side by side; it is the input of the pair classifier

The pair classifier ends with the **logits** `[1, K, 2]`, the input of the final softmax.

In the hybrid model the pair building runs on the FPGA too, in a custom C++ kernel that is provided: you will not write it.

In [ ]:
# Visualize the two cuts (no changes needed)
plot_cuts(save_to=FIGURE_DIR / "cuts.png")
print("Plot saved as 'work/figures/cuts.png'")

We find the three tensors from the **structure** of the graph, not from their names: names change from one export to the next, the structure does not.

### Task 3.1: The Logits
The output of the model is computed by the final `Softmax`. The logits are its input.

In [ ]:
# TODO: find the name of the logits: the input of the node that computes the output of the model

# Hint: the name of the model output is model.graph.output[0].name
# Hint: producer_of gives the node that computes it; node.input is the list of the names of its inputs

softmax_node = None  # Replace with correct code
logits_name = None   # Replace with correct code

print(f"Final node: {softmax_node.op_type}")
print(f"Logits    : {logits_name}")

### Task 3.2: The Pair Features
Walk back from the logits through the pair classifier (`MatMul`, `Add`, `Relu`), node by node, until you reach the `Concat` that puts the two embeddings of each pair side by side. Its output is the pair features. At each step, go to the input of the node that is **not** a weight: the weights are initializers.

In [ ]:
# TODO: walk back from the logits to the Concat, collecting the nodes of the pair classifier

# Hint: inside the loop: put node at the front of mlp_nodes (mlp_nodes.insert(0, node)),
# Hint: then set tensor_name to the input of node that is not in weight_names

weight_names = {initializer.name for initializer in model.graph.initializer}   # weights and constants

mlp_nodes = []               # the nodes of the pair classifier, in order
tensor_name = logits_name    # start from the logits...
while producer_of[tensor_name].op_type != "Concat":    # ...and walk back until the Concat
    node = producer_of[tensor_name]
    # YOUR CODE HERE
    break  # Replace with correct code

concat_node = producer_of[tensor_name]
pair_features_name = concat_node.output[0]

print(f"Pair classifier: {' -> '.join(node.op_type for node in mlp_nodes)}")
print(f"Pair features  : {pair_features_name}  (output of {concat_node.op_type})")

### Task 3.3: The Embeddings
The encoder ends with a `LayerNormalization` (the second normalisation of the second Transformer layer): its output is the embeddings.

In [ ]:
# TODO: find the name of the embeddings: the output of the last LayerNormalization node of the graph

# Hint: the nodes in model.graph.node are in execution order
# Hint: keep the nodes with node.op_type == 'LayerNormalization', take the last one, and its first output

embeddings_name = None  # Replace with correct code

readers = [node for node in model.graph.node if embeddings_name in node.input]
print(f"Embeddings: {embeddings_name}")
print(f"Read by   : {[f'{node.op_type} {node.name}' for node in readers]}")

The embeddings are read by exactly two `Gather` nodes: with the pair indices, they pick the embeddings of hit *i* and of hit *j* of every pair, and the `Concat` puts them side by side. This is the **pair building**, between the two cuts.

### Task 3.4: The Shapes at the Cuts
ONNX can infer the shape of every tensor of the graph. `N` and `K` stay symbolic (`unk__..` is a dynamic axis the exporter did not name).

In [ ]:
# Infer the shapes of all the tensors, and print the ones at the cuts (no changes needed)
inferred_model = onnx.shape_inference.infer_shapes(model)
inferred_shapes = {}
for value_info in list(inferred_model.graph.value_info) + list(inferred_model.graph.output):
    inferred_shapes[value_info.name] = shape_of(value_info)

for label, name in [("embeddings", embeddings_name), ("pair features", pair_features_name), ("logits", logits_name)]:
    print(f"{label:14s} {name:40s} {inferred_shapes.get(name, '(not inferred)')}")

### Task 3.5: Parameters and Work
How big is each part? The cell below counts the parameters (the numbers learned in training) of the encoder and of the pair classifier, and compares them with their work for a median event (51 hits, 1,326 pairs).

In [ ]:
# Count the parameters of the two parts, and compare them with their work (no changes needed)
weights = {initializer.name: numpy_helper.to_array(initializer) for initializer in model.graph.initializer}
mlp_node_names = {node.name for node in mlp_nodes}
parameters = {"encoder": 0, "pair classifier": 0}
already_counted = set()
for node in model.graph.node:
    part = "pair classifier" if node.name in mlp_node_names else "encoder"
    for name in node.input:
        is_parameter = name in weights and weights[name].dtype == np.float32 and weights[name].size > 1
        if is_parameter and name not in already_counted:
            parameters[part] += weights[name].size
            already_counted.add(name)
for part, count in parameters.items():
    print(f"{part:16s} {count:7,d} parameters")

plot_parameters_and_work(parameters["encoder"], parameters["pair classifier"], 51,
                         save_to=FIGURE_DIR / "parameters_and_work.png")
print("Plot saved as 'work/figures/parameters_and_work.png'")

The pair classifier has only a fifth of the parameters, but it does most of the work, because it runs once for every pair. This is why it is the part worth moving to the FPGA.

## Part 4: Run the Full Model
### Task 4.1: The Pair List
The model scores the pairs listed in `pair_indices`. The training code lists all the pairs *(i, j)* with *i ≤ j* with `itertools.combinations_with_replacement(range(N), 2)`: (0,0), (0,1), ..., (0,N−1), (1,1), (1,2), ... These are the cells of the upper triangle of an *N × N* matrix, diagonal included, row by row.

In [ ]:
# TODO: write a function that returns all the pairs (i, j) with i <= j, as an integer array [K, 2]

# Hint: np.triu_indices(n) returns two arrays: the row and the column indices of the upper triangle of an n x n
#       matrix, diagonal included, row by row
# Hint: np.stack([rows, columns], axis=1) puts them side by side; the model wants int64: use .astype(np.int64)

def pair_list(n_hits):
    # YOUR CODE HERE
    return None  # Replace with correct code

# Check: the same pairs, in the same order, as the training code
for n in (1, 2, 7, 51):
    assert np.array_equal(pair_list(n), np.array(list(combinations_with_replacement(range(n), 2))))
print(f"pair_list(4) =\n{pair_list(4)}")
print(f"An event with 51 hits has {len(pair_list(51))} pairs")

We now load the 400 events of the test set, as the tracking code prepares them (the raw hits are first clustered), and take a median event.

In [ ]:
# Load the test events and pick a median event (no changes needed)
test_events = load_test_events(CONFIG, model_config)
hits_per_event = np.array([len(event["features"]) for event in test_events])
median_event = test_events[np.argsort(hits_per_event)[len(hits_per_event) // 2]]
hit_features = median_event["features"]
pairs = pair_list(len(hit_features))
print(f"{len(test_events)} test events, median number of hits {np.median(hits_per_event):.0f}")
print(f"Median event: {median_event['name']}, data {hit_features.shape}, pair_indices {pairs.shape}")

### Task 4.2: Run the Model with ONNX Runtime
ONNX Runtime runs an ONNX model on the CPU. Load the trained model in an inference session, and run it on the median event.

In [ ]:
# TODO: run the full model on the median event with ONNX Runtime

# Hint: onnxruntime.InferenceSession(str(path), providers=['CPUExecutionProvider']) loads a model
# Hint: session.run(None, {input name: array, ...}) runs it and returns the list of its outputs (here only one):
#       the inputs are 'data' (hit_features) and 'pair_indices' (pairs)

full_session = None   # Replace with correct code
probabilities = None  # Replace with correct code: the output of the model, [1, K, 2]

print(f"Output: {probabilities.shape}")
print("First rows (i, j, p0, p1):")
print(np.column_stack([pairs[:5], probabilities[0, :5]]).round(4))
print(f"Pairs with p1 > {THRESHOLD}: {(probabilities[0, :, 1] > THRESHOLD).sum()} of {len(pairs)}")

### Task 4.3: Visualize the Output
Row *r* of the output is the pair in row *r* of `pair_indices`. Putting each p₁ back at its place *(i, j)* gives a picture of the event: dark cells are the pairs the model thinks are edges, i.e. consecutive hits of the same particle.

In [ ]:
# Visualize the probabilities of the median event (no changes needed)
plot_probability_matrix(probabilities[0], pairs, THRESHOLD, title=f" ({len(hit_features)} hits)",
                        save_to=FIGURE_DIR / "probability_matrix.png")
print("Plot saved as 'work/figures/probability_matrix.png'")

The colour scale goes up to the largest p₁ of the event: for this model p₁ stays below about 0.55, and the threshold (red) is 0.1. Most pairs have p₁ close to 0: of all the pairs of an event, only a few are consecutive hits of the same particle. The threshold decides which pairs become **edges**, and the edges are what the track building reads. The diagonal holds the pairs *(i, i)* of a hit with itself, which the model scores too.

## Part 5: Split the Model
`onnx.utils.Extractor` copies the part of a graph between some input tensors and some output tensors, with the initializers it needs. It needs to know the shape of the tensors at the cuts: the cell below adds the ones that shape inference could not find.

In [ ]:
# Prepare the extractor (no changes needed)
cut_shapes = {embeddings_name: [1, "num_nodes", 32],
              pair_features_name: [1, "num_pairs", 64],
              logits_name: [1, "num_pairs", 2]}
for name, shape in cut_shapes.items():
    if name not in inferred_shapes:
        inferred_model.graph.value_info.append(helper.make_tensor_value_info(name, TensorProto.FLOAT, shape))
extractor = onnx.utils.Extractor(inferred_model)

### Task 5.1: Extract the Two Parts
- the **encoder** goes from the input `data` to the embeddings
- the **pair classifier** goes from the pair features to the logits

In [ ]:
# TODO: extract the encoder and the pair classifier from the full model

# Hint: extractor.extract_model(input_names, output_names) returns the sub-model between two lists of tensor names
# Hint: use the names you found in Part 3: embeddings_name, pair_features_name, logits_name

encoder = None    # Replace with correct code
pair_head = None  # Replace with correct code: the pair classifier

print(f"Encoder        : {len(encoder.graph.node)} nodes, {' '.join(node.op_type for node in encoder.graph.node[:6])} ...")
print(f"Pair classifier: {len(pair_head.graph.node)} nodes, {' -> '.join(node.op_type for node in pair_head.graph.node)}")

Two finishing touches make the two parts easier to use: the output of the encoder is renamed `embeddings`, and the pair classifier is wrapped so that it takes `pair_features [K, 64]` and returns `logits [K, 2]`, without the batch axis. Then the two parts are checked and saved.

In [ ]:
# Finishing touches, check and save the two parts (no changes needed)
encoder.graph.node.append(helper.make_node("Identity", [embeddings_name], ["embeddings"], name="rename_embeddings"))
del encoder.graph.output[:]
encoder.graph.output.append(helper.make_tensor_value_info("embeddings", TensorProto.FLOAT, [1, "num_nodes", 32]))
del encoder.graph.input[:]
encoder.graph.input.append(helper.make_tensor_value_info("data", TensorProto.FLOAT, ["num_nodes", 4]))

axis_zero = numpy_helper.from_array(np.array([0], dtype=np.int64), name="axis_zero")
add_batch_axis = helper.make_node("Unsqueeze", ["pair_features", "axis_zero"], [pair_features_name], name="add_batch_axis")
drop_batch_axis = helper.make_node("Squeeze", [logits_name, "axis_zero"], ["logits"], name="drop_batch_axis")
classifier_nodes = list(pair_head.graph.node)
del pair_head.graph.node[:]
pair_head.graph.node.extend([add_batch_axis] + classifier_nodes + [drop_batch_axis])
pair_head.graph.initializer.append(axis_zero)
del pair_head.graph.input[:]
pair_head.graph.input.append(helper.make_tensor_value_info("pair_features", TensorProto.FLOAT, ["num_pairs", 64]))
del pair_head.graph.output[:]
pair_head.graph.output.append(helper.make_tensor_value_info("logits", TensorProto.FLOAT, ["num_pairs", 2]))

for part, file_name in ((encoder, "encoder.onnx"), (pair_head, "pair_head.onnx")):
    onnx.checker.check_model(part, full_check=True)
    onnx.save(part, ONNX_DIR / file_name)
    inputs = [(value.name, shape_of(value)) for value in part.graph.input]
    outputs = [(value.name, shape_of(value)) for value in part.graph.output]
    print(f"{file_name:15s} {len(part.graph.node):3d} nodes, inputs {inputs}, outputs {outputs}")

encoder_session = onnxruntime.InferenceSession(str(ONNX_DIR / "encoder.onnx"), providers=["CPUExecutionProvider"])
pair_head_session = onnxruntime.InferenceSession(str(ONNX_DIR / "pair_head.onnx"), providers=["CPUExecutionProvider"])

### Task 5.2: Check the Split
The split model computes an event in four steps: the encoder gives the embeddings; the pair building puts the embeddings of hit *i* and hit *j* side by side for every pair (in the hybrid model this is done by the provided kernel; here we do it with numpy); the pair classifier gives the logits; the softmax gives the probabilities.

In [ ]:
# TODO: compute the probabilities of an event with the split model

# Hint: encoder_session.run(None, {'data': hits})[0] gives the embeddings [1, N, 32]: take [0] to get [N, 32]
# Hint: embeddings[pairs[:, 0]] are the embeddings of the first hit of every pair, [K, 32];
#       np.concatenate([..., ...], axis=1) puts the two hits side by side: [K, 64]
# Hint: pair_head_session.run(None, {'pair_features': ...})[0] gives the logits [K, 2], and softmax(logits)
#       (from utils) the probabilities

def split_model_probabilities(hits, pairs):
    # YOUR CODE HERE
    return None  # Replace with correct code: the probabilities [K, 2]

split_probabilities = split_model_probabilities(hit_features, pairs)
print(f"Median event: split model {split_probabilities.shape}, "
      f"largest difference from the full model {np.abs(split_probabilities - probabilities[0]).max():.1e}")

Now on all the test events: the split model must give the same probabilities as the full model, and above all the same decisions at the threshold.

In [ ]:
# Compare the split and the full model on all the test events (no changes needed)
full_p1, split_p1 = [], []
for event in test_events:
    event_pairs = pair_list(len(event["features"]))
    full_p1.append(full_session.run(None, {"data": event["features"], "pair_indices": event_pairs})[0][0, :, 1])
    split_p1.append(split_model_probabilities(event["features"], event_pairs)[:, 1])
full_p1 = np.concatenate(full_p1)
split_p1 = np.concatenate(split_p1)

decision_changes = int(((split_p1 > THRESHOLD) != (full_p1 > THRESHOLD)).sum())
print(f"{len(test_events)} events, {len(full_p1):,} pairs")
print(f"Largest difference in p1: {np.abs(split_p1 - full_p1).max():.1e}")
print(f"Decisions changed at the threshold: {decision_changes}")

plot_split_check(full_p1, split_p1, save_to=FIGURE_DIR / "split_check.png")
print("Plot saved as 'work/figures/split_check.png'")

The split changes nothing: more than half of the probabilities are exactly the same, the others differ by less than 10⁻⁷ (the last digits of a float: the embeddings and the logits are identical, but the softmax is now computed by numpy instead of ONNX Runtime), and no decision changes.

## Part 6: Save the Inputs of Exercise 2
In Exercise 2 you will convert the pair classifier with _hls4ml_. You need its weights, and some real pair features to choose and check its fixed-point precision.

### Task 6.1: The Weights of the Pair Classifier
The weights are the initializers of `pair_head.onnx`. ONNX stores each weight matrix as `[inputs, outputs]`.

In [ ]:
# TODO: read the initializers of pair_head as numpy arrays

# Hint: the initializers are in pair_head.graph.initializer; initializer.name is the name of one of them
# Hint: numpy_helper.to_array(initializer) converts it to a numpy array

pair_head_initializers = None  # Replace with correct code: {initializer name: numpy array}

for name, array in pair_head_initializers.items():
    print(f"{name:20s} {str(array.shape):9s} {array.dtype}")

Two weight matrices (64 × 64 and 64 × 2), two biases (64 and 2), and the `axis_zero` constant of the wrapping. The cell below picks the two layers in order, checks that they are exactly the weights of the full model, and saves them. It also saves the pair features of 100 events of the validation set (to choose the precision) and of 100 events of the test set (to check it), with their logits.

In [ ]:
# Save the weights and the reference pairs for Exercise 2 (no changes needed)
layer1, layer2 = hi_common.mlp_layers_from_onnx(pair_head)        # the two Linear layers, in order
for layer in (layer1, layer2):
    assert np.array_equal(layer["W"], weights[layer["W_name"]]) and np.array_equal(layer["b"], weights[layer["b_name"]])
classifier_layers = {"W1": layer1["W"], "b1": layer1["b"], "W2": layer2["W"], "b2": layer2["b"]}
np.savez(DATA_DIR / "pair_head_weights.npz", **classifier_layers)
print("Saved pair_head_weights.npz:", {name: array.shape for name, array in classifier_layers.items()})

reference = {}
for split in ("validation", "test"):
    all_pair_features = []
    for event in hi_common.load_events(CONFIG, model_config, split, 100):
        event_pairs = pair_list(len(event["features"]))
        embeddings = encoder_session.run(None, {"data": event["features"]})[0][0]
        all_pair_features.append(np.concatenate([embeddings[event_pairs[:, 0]], embeddings[event_pairs[:, 1]]], axis=1))
    pair_features = np.ascontiguousarray(np.concatenate(all_pair_features), dtype=np.float32)
    logits = pair_head_session.run(None, {"pair_features": pair_features})[0]
    np.savez(DATA_DIR / f"pairs_{split}.npz", X=pair_features, logits=logits)
    reference[split] = pair_features
    print(f"Saved pairs_{split}.npz: {len(pair_features):,} pairs from 100 events")

### Task 6.2: The Numbers the FPGA Will See
On the FPGA every number is a **fixed-point** number with a fixed number of bits, so we must know the range of the values: the inputs of the pair classifier and its weights.

In [ ]:
# Visualize the range of the inputs and of the weights of the pair classifier (no changes needed)
plot_value_ranges(reference["validation"], classifier_layers, save_to=FIGURE_DIR / "value_ranges.png")
print("Plot saved as 'work/figures/value_ranges.png'")

The pair features stay within about ±4.2, the weights and biases within ±0.33. In Exercise 2 you will use these ranges to choose how many bits the FPGA needs, before and after the binary point.

| file (in `work/`) | content |
|---|---|
| `onnx/encoder.onnx` | the encoder: `data [N, 4]` → `embeddings [1, N, 32]` (stays on the CPU) |
| `onnx/pair_head.onnx` | the pair classifier: `pair_features [K, 64]` → `logits [K, 2]` (goes to the FPGA) |
| `data/pair_head_weights.npz` | its weights and biases: `W1 [64, 64]`, `b1`, `W2 [64, 2]`, `b2` |
| `data/pairs_validation.npz`, `data/pairs_test.npz` | real pair features `X [M, 64]` and their logits |

In **Exercise 2** you will convert the pair classifier with _hls4ml_, choose its fixed-point precision and estimate its FPGA resources. The pair building is done by the custom kernel that is provided: you will see how it is connected to the _hls4ml_ network.